# Logistic Regression from scratch 
Covers: why linear regression fails for classification, sigmoid, BCE loss (vs MSE),
gradient descent, decision boundary, metrics (confusion matrix, precision, recall, F1),
threshold tuning / ROC, softmax regression for multi-class.

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(7)

## 1. Why linear regression fails: pass/fail vs hours studied (from the doc)

In [ ]:
hours = np.arange(1, 11, dtype=float)
passed = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1], dtype=float)
A = np.c_[np.ones(10), hours]
lin = np.linalg.solve(A.T @ A, A.T @ passed)
hh = np.linspace(-2, 14, 100)
print("Linear regression predicts at hours=-2:", (lin[0] + lin[1] * -2).round(2),
      " and at 14:", (lin[0] + lin[1] * 14).round(2), "-> outside [0,1]")

## 2. Sigmoid  s(z) = 1/(1+e^-z)   (numerically stable version)

In [ ]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1 / (1 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1 + ez)
    return out

print("sigmoid(0)=", sigmoid(0.0), " sigmoid(large)=", sigmoid(np.array([50.0, -50.0])))

## 3. Loss functions: BCE = -mean[y log p + (1-y) log(1-p)]
Plus the gradient: dJ/dw = (1/m) X^T (p - y), dJ/db = mean(p - y) (same form as linear regression!).

In [ ]:
def bce_loss(y, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def mse_loss(y, p):
    return np.mean((y - p) ** 2)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(hours, passed, c='k'); ax[0].plot(hh, lin[0] + lin[1] * hh, 'r', label='linear regression')
ax[0].plot(hh, sigmoid(1.2 * (hh - 4.5)), 'g', label='sigmoid'); ax[0].set_ylim(-0.5, 1.5)
ax[0].axhline(0, c='gray', lw=.5); ax[0].axhline(1, c='gray', lw=.5); ax[0].legend(); ax[0].set_title('Linear fails, sigmoid works')
zz = np.linspace(-8, 8, 200); ax[1].plot(zz, sigmoid(zz)); ax[1].axhline(.5, c='gray', ls='--'); ax[1].set_title('Sigmoid')
pp = np.linspace(0.001, 0.999, 300)
ax[2].plot(pp, -np.log(pp), label='y=1: -log(p)'); ax[2].plot(pp, -np.log(1 - pp), label='y=0: -log(1-p)')
ax[2].set_ylim(0, 5); ax[2].legend(); ax[2].set_title('BCE penalty')
plt.tight_layout(); plt.savefig('plots/01_sigmoid_bce.png', dpi=120); plt.close()

## 4. Model class: fit with gradient descent (optional L2), predict_proba, predict

In [ ]:
class LogisticRegression:
    def __init__(self, lr=0.1, n_iters=2000, l2=0.0):
        self.lr, self.n_iters, self.l2 = lr, n_iters, l2

    def fit(self, X, y):
        m, k = X.shape
        self.w, self.b = np.zeros(k), 0.0
        self.history = []
        for _ in range(self.n_iters):
            p = sigmoid(X @ self.w + self.b)
            err = p - y
            self.w -= self.lr * (X.T @ err / m + self.l2 * self.w)
            self.b -= self.lr * err.mean()
            self.history.append(bce_loss(y, p))
        return self

    def predict_proba(self, X):
        return sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

# 1-D check on the hours data
m1 = LogisticRegression(lr=0.3, n_iters=5000).fit(hours.reshape(-1, 1), passed)
print(f"1-D model: w={m1.w[0]:.3f}, b={m1.b:.3f}, boundary at hours={-m1.b / m1.w[0]:.2f}")

## 5. Two-feature dataset, standardisation, train/test split, decision boundary
Boundary: w1 x1 + w2 x2 + b = 0  (where p = 0.5).

In [ ]:
n = 400
X0 = rng.normal([-1, -1], 1.2, size=(n // 2, 2)); X1 = rng.normal([1.5, 1], 1.2, size=(n // 2, 2))
X = np.r_[X0, X1]; y = np.r_[np.zeros(n // 2), np.ones(n // 2)]
idx = rng.permutation(n); tr, te = idx[:300], idx[300:]
mu, sd = X[tr].mean(0), X[tr].std(0)
Xs = (X - mu) / sd
model = LogisticRegression(lr=0.5, n_iters=1500).fit(Xs[tr], y[tr])
print("weights:", model.w.round(3), "bias:", round(model.b, 3))

def plot_boundary(ax, model, X, y, title):
    gx, gy = np.meshgrid(np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 200),
                         np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 200))
    P = model.predict_proba(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    cs = ax.contourf(gx, gy, P, levels=20, cmap='RdBu_r', alpha=.6)
    ax.contour(gx, gy, P, levels=[0.5], colors='k')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=15); ax.set_title(title)
    return cs

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
plot_boundary(ax[0], model, Xs[tr], y[tr], 'Decision boundary (train, standardised)')
ax[1].plot(model.history); ax[1].set_xlabel('iteration'); ax[1].set_ylabel('BCE'); ax[1].set_title('Loss curve')
plt.tight_layout(); plt.savefig('plots/02_decision_boundary.png', dpi=120); plt.close()

## 6. Metrics: confusion matrix, accuracy, precision, recall, F1

In [ ]:
def confusion(y, yhat):
    tp = int(np.sum((y == 1) & (yhat == 1))); tn = int(np.sum((y == 0) & (yhat == 0)))
    fp = int(np.sum((y == 0) & (yhat == 1))); fn = int(np.sum((y == 1) & (yhat == 0)))
    return tp, fp, fn, tn

def metrics(y, yhat):
    tp, fp, fn, tn = confusion(y, yhat)
    acc = (tp + tn) / len(y)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1)

yhat = model.predict(Xs[te]); tp, fp, fn, tn = confusion(y[te], yhat)
print(f"Confusion  TP={tp} FP={fp} FN={fn} TN={tn}")
print({k: round(v, 3) for k, v in metrics(y[te], yhat).items()})
fig, ax = plt.subplots(figsize=(3.6, 3.4)); ax.imshow([[tn, fp], [fn, tp]], cmap='Blues')
for (i, j), v in zip([(0, 0), (0, 1), (1, 0), (1, 1)], [tn, fp, fn, tp]): ax.text(j, i, v, ha='center', va='center', fontsize=14)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title('Confusion matrix')
plt.tight_layout(); plt.savefig('plots/03_confusion_matrix.png', dpi=120); plt.close()

## 7. Threshold tuning: precision/recall trade-off and ROC curve (AUC by trapezoid rule)

In [ ]:
proba = model.predict_proba(Xs[te])
ths = np.linspace(0.01, 0.99, 99)
P_, R_, F_ = zip(*[(lambda d: (d['precision'], d['recall'], d['f1']))(metrics(y[te], (proba >= t).astype(int))) for t in ths])
tpr = np.array([np.mean(proba[y[te] == 1] >= t) for t in np.r_[1.01, ths[::-1], -0.01]])
fpr = np.array([np.mean(proba[y[te] == 0] >= t) for t in np.r_[1.01, ths[::-1], -0.01]])
auc = float(np.sum(np.diff(fpr) * (tpr[1:] + tpr[:-1]) / 2))  # trapezoid rule
print(f"AUC = {auc:.3f}; recall at t=0.2: {R_[19]:.2f}, at t=0.8: {R_[79]:.2f}")
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ths, P_, label='precision'); ax[0].plot(ths, R_, label='recall'); ax[0].plot(ths, F_, label='F1')
ax[0].set_xlabel('threshold'); ax[0].legend(); ax[0].set_title('Threshold trade-off')
ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], 'k--'); ax[1].set_title(f'ROC (AUC={auc:.3f})')
ax[1].set_xlabel('FPR'); ax[1].set_ylabel('TPR')
plt.tight_layout(); plt.savefig('plots/04_threshold_roc.png', dpi=120); plt.close()

## 8. Why BCE, not MSE: loss surfaces over (w, b)
BCE is convex; MSE with sigmoid is non-convex with flat regions.

In [ ]:
xh, yh = hours - hours.mean(), passed
ws, bs = np.linspace(-3, 6, 80), np.linspace(-6, 6, 80)
Jb = np.array([[bce_loss(yh, sigmoid(w * xh + b)) for w in ws] for b in bs])
Jm = np.array([[mse_loss(yh, sigmoid(w * xh + b)) for w in ws] for b in bs])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, J, t in zip(ax, [Jb, Jm], ['BCE surface', 'MSE surface']):
    c = a.contourf(ws, bs, J, 25); plt.colorbar(c, ax=a); a.set_title(t); a.set_xlabel('w'); a.set_ylabel('b')
plt.tight_layout(); plt.savefig('plots/05_loss_surfaces.png', dpi=120); plt.close()

## 9. Softmax regression (multi-class)
softmax(z_i) = e^{z_i}/sum e^{z_j}; loss = categorical cross-entropy; gradient = X^T (P - Y)/m.

In [ ]:
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)      # stability
    E = np.exp(Z); return E / E.sum(axis=1, keepdims=True)

class SoftmaxRegression:
    def __init__(self, lr=0.5, n_iters=1000):
        self.lr, self.n_iters = lr, n_iters
    def fit(self, X, y, k):
        m, d = X.shape; Y = np.eye(k)[y]
        self.W, self.b = np.zeros((d, k)), np.zeros(k); self.history = []
        for _ in range(self.n_iters):
            P = softmax(X @ self.W + self.b)
            self.W -= self.lr * X.T @ (P - Y) / m; self.b -= self.lr * (P - Y).mean(0)
            self.history.append(-np.mean(np.sum(Y * np.log(P + 1e-12), axis=1)))
        return self
    def predict_proba(self, X): return softmax(X @ self.W + self.b)
    def predict(self, X): return self.predict_proba(X).argmax(1)

centers = np.array([[0, 2], [2, -1], [-2, -1]])
Xm = np.concatenate([rng.normal(c, 0.8, size=(100, 2)) for c in centers]); ym = np.repeat([0, 1, 2], 100)
sm = SoftmaxRegression().fit(Xm, ym, 3)
print("softmax accuracy:", (sm.predict(Xm) == ym).mean().round(3), " probs sum to", sm.predict_proba(Xm[:3]).sum(1))
gx, gy = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-4, 5, 200))
Z = sm.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
plt.figure(figsize=(5, 4.5)); plt.contourf(gx, gy, Z, alpha=.3, cmap='viridis'); plt.scatter(Xm[:, 0], Xm[:, 1], c=ym, cmap='viridis', edgecolor='k', s=15)
plt.title('Softmax regression'); plt.tight_layout(); plt.savefig('plots/06_softmax.png', dpi=120); plt.close()